# Training RT-DETR Cell Instance Detection

## Preparations
### Import required libraries

In [ ]:
import os
import sys
import time
import pickle
import json
from typing import Tuple, Union, List, Dict, Final

import numpy as np
from PIL import Image
import cv2
import torch
import pycocotools
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask_util
import albumentations as A
sys.path.append('references/detection')

## Pre-processing configurations
The following configurations are used for pre-processing the images during the training. 

In [ ]:
# probability of adding random blur and salt-and-pepper or additive gaussian noise to the training images
# see the image Transforms section below
P_NOISE = 0.25

# the lower and upper bounds for random scaling the images (and annotated masks) for training augmentation
MIN_RANDOM_SCALE = 0.7
MAX_RANDOM_SCALE = 1.0

# model input image size (should be square)
MODEL_INPUT_SIZE: Final[int] = 640

SEGMENTATION_MODEL: Final[bool] = False

### Transforms
We are using `Albumentations` package for all image and annotation augmentations.  

In [ ]:
def get_transform(train: bool = True) -> A.core.composition.Compose:
    # no resizing is needed as the images are already prepared as 640 x 640 (the same as the model's input size)
    if train:
        # random noise addition and random scale as defined above, 
        # we call these before PILToTensor as these classes 
        # operates on PIL images
        trsfms = [
            A.RandomScale(scale_limit=(MIN_RANDOM_SCALE - 1.0, MAX_RANDOM_SCALE - 1.0), p=1.0),
            A.PadIfNeeded(min_height = MODEL_INPUT_SIZE, min_width = MODEL_INPUT_SIZE, position = 'random'),
            A.RandomRotate90(),
            A.Perspective(p=0.1),
            A.RandomBrightnessContrast(p=P_NOISE),
            A.HueSaturationValue(p=P_NOISE),
        ]        
    else:
        # test set (only convert the bboxes)
        trsfms = [A.NoOp()]

    return A.Compose(trsfms, bbox_params=A.BboxParams(format="coco", label_fields=["category"], clip=True, min_area=1))

## Data Model
The dataset class as well as the training and evaluation scripts are adopted from the RT-DETR fine-tuning tuturial Notebook here:  https://github.com/NielsRogge/Transformers-Tutorials/blob/master/RT-DETR/Fine_tune_RT_DETR_on_a_custom_dataset.ipynb. This Notebook provides the dataset class for the pre-processed annotated data (resized and cropped) as prepared for training our YOLO model (640 x 640 crops). 

NOTE: The labels (class IDs) for both YOLO and RT-DETR models starts with 0.

### Dataset for already pre-processed annotated data
This dataset is built by passing the location of the pre-processed images and bounding box folders (labels). The code below assumes the images and the corresponding annotation files use the same name. For each image with name `sample_name` (saved as .jpg or .png) under the images folder, there should be an annotation file with the same name and .txt extension (`sample_name.txt`) under the annotation folder. For a faster training, pre-process the images and use the dataset below.

In [ ]:
class CellMaskDataset(torch.utils.data.Dataset):
    def __init__(self, 
                 images_path: str,
                 annotations_path: str, 
                 instance_segmentation: bool,
                 percentage_to_expand_bbox_boundaries: float, 
                 transforms: A.core.composition.Compose) -> None:
        
        self.images_path = images_path
        self.annotations_path = annotations_path
        self.instance_segmentation = instance_segmentation
        self.transforms = transforms
        # load all images and masks
        # the assumption is the image and its mask annotation use the same name
        self.imgs = list(sorted(os.listdir(images_path)))
        self.annotations = list(sorted(os.listdir(annotations_path)))
        self.percentage_to_expand_bbox_boundaries = percentage_to_expand_bbox_boundaries
        

        if len(self.imgs) != len(self.annotations):
            print("[ERROR]: The list of images and annotations are not consistent")
            return
        
        for i, img_filename in enumerate(self.imgs):
            # drop the image/mask filename extension 
            # (anything after the last '.' in the filename is considered as extension)
            img_name = ".".join(img_filename.strip().split('.')[:-1])
            annots_name = ".".join(self.annotations[i].strip().split('.')[:-1])
            if img_name != annots_name:
                print("[ERROR]: Inconsistent annotations file :{} found for image file: {}".format(annots_name, img_name))
     

    def __getitem__(self, idx: int):

        # a unique image identifier
        image_id: int = idx
        # load images and masks
        img_path = os.path.join(self.images_path, self.imgs[idx])
        annots_path = os.path.join(self.annotations_path, self.annotations[idx])
        # read the image, do not change the format
        # the processed images are all 8-bit (bit-depth)
        # the model expect the image in RGB format, convert grayscale images to RGB
        img = np.array(Image.open(img_path).convert('RGB'))
            
        image_height, image_width = img.shape[:2]
        # annotations
        # boxes should be in COCO format (xtl, ytl, w, h) because this is the format the model and the
        # albumentations expect
        boxes: List[List[int]] = [] 
        labels: List[int] = []
        masks: List[np.ndarray] = []
        
        with open(annots_path,'r') as annot_file:
            num_annotations = 0
            for line in annot_file:
                fields = line.strip().split(' ')
                if self.instance_segmentation:
                    # convert the label to an integer from string
                    label = int(fields[0])
                    # convert the polygon points fron strings to floats
                    points = np.array([float(point) for point in fields[1:]])
                    # rearrange them in (x, y)
                    polygon_points = np.reshape(points, (int(len(points) / 2), 2)) * np.array([image_width, image_height], dtype=float)
                    # convert the points to integers and use CV2 contours format
                    polygon_points = np.expand_dims(polygon_points.astype(int), axis=1)
                    # bounding box of the mask contour
                    (xtl, ytl, w, h) = cv2.boundingRect(polygon_points)
                    # skip zero area boxes
                    if w <= 0 or h <= 0:
                        continue
                    xbr = xtl + w 
                    ybr = ytl + h
                    
                    # the mask
                    mask: np.ndarray = np.zeros((image_height, image_width), np.uint8)
                    # create the mask for the object (the mask value is set to 1 for the object)
                    cv2.drawContours(mask, [polygon_points], 0, 1, -1)
                    masks.append(mask)
                else:
                    (label, center_x, center_y, w, h) = fields
                    xtl = int((float(center_x) - float(w) / 2.0) * image_width)
                    ytl = int((float(center_y) - float(h) / 2.0) * image_height)
                    xbr = int((float(center_x) + float(w) / 2.0) * image_width)
                    ybr = int((float(center_y) + float(h) / 2.0) * image_height)
                    if xtl >= xbr or ytl >= ybr:
                        continue
                    # convert the label to an integer from string
                    label = int(label)

                # expand the bounding boxes if necessary
                delta_x = int(self.percentage_to_expand_bbox_boundaries * (xbr - xtl) / 2)
                delta_y = int(self.percentage_to_expand_bbox_boundaries * (ybr - ytl) / 2)
                # # expand by one pixel on each side at least to cover boundaries
                delta_x = max(1, delta_x)
                delta_y = max(1, delta_y)
            
                xtl = max(0, xtl - delta_x)
                ytl = max(0, ytl - delta_y)
                xbr = min(image_width, xbr + delta_x)
                ybr = min(image_height, ybr + delta_y)
                boxes.append([xtl, ytl, xbr - xtl, ybr - ytl])
                labels.append(label)
                num_annotations += 1
        
        # convert boxes to a numpy array
        boxes: np.ndarray = np.array(boxes)
        
        # apply augmentations, the second condition should not happen (all preprocessed images should at least have one object)
        if self.transforms and len(boxes) > 0:
            if self.instance_segmentation:
                # TODO: check to make sure the below mask transform would work
                transformed = self.transforms(image=img, bboxes=boxes, masks=masks, category=labels)
                img = transformed["image"]
                boxes = transformed["bboxes"]
                masks = transformed["masks"]
                labels = transformed["category"]
            else:
                transformed = self.transforms(image=img, bboxes=boxes, category=labels)
                img = transformed["image"]
                boxes = transformed["bboxes"]
                labels = transformed["category"]
        
        # reformat annotations
        annotations: List[dict] = []
        for i, bbox in enumerate(boxes):
            formatted_annotation = {
                "image_id": image_id,
                "category_id": labels[i],
                "bbox": bbox,
                "iscrowd": 0,
                "area": bbox[2] * bbox[3],
            }
            if self.instance_segmentation:
                formatted_annotation["mask"] = masks[i]
            annotations.append(formatted_annotation)

        return img, {"image_id": image_id, "annotations": annotations,}

    def __len__(self):
        return len(self.imgs)


#### A function to convert our dataset to COCO dataset format
This function is needed for efficient evaluation. Similar to the dataset class, the labels (class IDs) should start from 0. 

In [ ]:
from tqdm import tqdm
def convert_to_coco_api(images_path: str, 
                        annotations_path: str, 
                        instance_segmentation: bool = False,
                        percentage_to_expand_bbox_boundaries: float = 0.0):
    # load all images and annotations
    # the assumption is the image and its annotation use the same name
    imgs = list(sorted(os.listdir(images_path)))
    annotations = list(sorted(os.listdir(annotations_path)))
    
    if len(imgs) != len(annotations):
        print("[ERROR]: The list of images and masks are not consistent")
        return False, {}
    
    for i, img_filename in enumerate(imgs):
        # drop the image/mask filename extension 
        # (anything after the last '.' in the filename is considered as extension)
        img_name = ".".join(img_filename.strip().split('.')[:-1])
        annots_name = ".".join(annotations[i].strip().split('.')[:-1])
        if img_name != annots_name:
            print("[ERROR]: Inconsistent annotations file :{} found for image file: {}".format(annots_name, img_name))
            return False, {}
    # the index for annotations starts at 1
    annots_id = 1
    categories = set()
    json_annotations = {"images": [], "categories": [], "annotations": []}
    for idx in tqdm(range(len(imgs))):
        # load the image
        img_path = os.path.join(images_path, imgs[idx])
        # read the image, we only read the image to get the size of it
        # so no need to change the format (BGR to RGB) or convert to PIL 
        opencv_img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        image_height, image_width = opencv_img.shape[:2]
        img_dict = {}
        img_dict["id"] = idx + 1
        img_dict["file_name"] = imgs[idx]
        img_dict["height"] = image_height
        img_dict["width"] = image_width
        json_annotations["images"].append(img_dict)


        # annotations
        boxes: List[List[int]] = []
        labels: List[int] = []
        masks: List[np.ndarray] = []
        # load the annotations (masks)
        annots_path = os.path.join(annotations_path, annotations[idx])
        
        with open(annots_path,'r') as annot_file:
            for line in annot_file:
                fields = line.strip().split(' ')
                if instance_segmentation:
                    # convert the label to an integer from string
                    label = int(fields[0])
                    # convert the polygon points fron strings to floats
                    points = np.array([float(point) for point in fields[1:]])
                    # rearrange them in (x, y)
                    polygon_points = np.reshape(points, (int(len(points) / 2), 2)) * np.array([image_width, image_height], dtype=float)
                    # convert the points to integers and use CV2 contours format
                    polygon_points = np.expand_dims(polygon_points.astype(int), axis=1)
                    # bounding box of the mask contour
                    (xtl, ytl, w, h) = cv2.boundingRect(polygon_points)
                    if w <= 0 or h <= 0:
                        continue
                    xbr = xtl + w 
                    ybr = ytl + h
                    # the mask
                    mask: np.ndarray = np.zeros((image_height, image_width), np.uint8)
                    # create the mask for the object (the mask value is set to 1 for the object)
                    cv2.drawContours(mask, [polygon_points], 0, 1, -1)
                    masks.append(mask)
                else:
                    (label, center_x, center_y, w, h) = fields
                    xtl = int((float(center_x) - float(w) / 2.0) * image_width)
                    ytl = int((float(center_y) - float(h) / 2.0) * image_height)
                    xbr = int((float(center_x) + float(w) / 2.0) * image_width)
                    ybr = int((float(center_y) + float(h) / 2.0) * image_height)
                    if xtl >= xbr or ytl >= ybr:
                        continue
                    # convert the label to an integer from string
                    label = int(label)

                # expand the bounding boxes if necessary
                delta_x = int(percentage_to_expand_bbox_boundaries * (xbr - xtl) / 2)
                delta_y = int(percentage_to_expand_bbox_boundaries * (ybr - ytl) / 2)
                # # expand by one pixel on each side at least to cover boundaries
                delta_x = max(1, delta_x)
                delta_y = max(1, delta_y)
            
                xtl = max(0, xtl - delta_x)
                ytl = max(0, ytl - delta_y)
                xbr = min(image_width, xbr + delta_x)
                ybr = min(image_height, ybr + delta_y)
                boxes.append([xtl, ytl, xbr, ybr])
                labels.append(label)
        
        for i, box in enumerate(boxes):
            record = {}
            record["image_id"] = idx + 1
            record['category_id'] = labels[i] 
            categories.add(record['category_id'])
            if instance_segmentation:
                record["segmentation"] = coco_mask_util.encode(np.asarray(masks[i], order="F"))
                record["segmentation"]['counts'] = record["segmentation"]['counts'].decode('utf8')
            xmin, ymin, xmax, ymax = [int(v) for v in box]
            #convert to xywh
            record['bbox'] = [xmin, ymin, xmax - xmin, ymax - ymin]
            record["area"] = (ymax - ymin) * (xmax - xmin)
            record["iscrowd"] = 0
            record["id"] = annots_id
                
            json_annotations["annotations"].append(record)
            annots_id += 1 
            
    json_annotations["categories"] = [{"id": i} for i in sorted(categories)]

    return True, json_annotations

#### Dataset prepration for option 3
----------------

Define LABEL_MAP, a mapping between class IDs and class names used in the annotations, with class IDs starting from 1  (0 is reseved for background). This is not needed for the dataset, but during the training. 

In [ ]:
# make sure these folders are generated in advance
# BASE_PATH = '/media/labuser/extra_drive/mlops/data_vault/training_dataset/all_100percent_dataset/20240306_mousepbmc-beads_10x_uncaged_yolo'
BASE_PATH = '/media/labuser/extra_drive/mlops/data_vault/training_dataset/all_datasets_0p25_suspension_10x_bf_yolo'
TRAIN_IMAGE_FOLDER = os.path.join(BASE_PATH, 'images', 'train')
TRAIN_MASK_FOLDER = os.path.join(BASE_PATH, 'labels', 'train')
TEST_IMAGE_FOLDER =os.path.join(BASE_PATH, 'images', 'test')
TEST_MASK_FOLDER = os.path.join(BASE_PATH, 'labels', 'test')

# mapping between the class IDs and class names for the annotated data 
# labels start from 0
LABEL_MAP = {0: 'cell', 1: 'bead', 2: 'soma', 3: 'cell-adhered'}
REVERESE_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

train_dataset = CellMaskDataset(images_path=TRAIN_IMAGE_FOLDER, 
                                annotations_path=TRAIN_MASK_FOLDER,
                                instance_segmentation = SEGMENTATION_MODEL,
                                percentage_to_expand_bbox_boundaries = 0.0,
                                transforms = get_transform(train=True))

# test_dataset = CellMaskDataset(images_path=TEST_IMAGE_FOLDER, 
#                                annotations_path=TEST_MASK_FOLDER,
#                                instance_segmentation = SEGMENTATION_MODEL,
#                                percentage_to_expand_bbox_boundaries = 0.0,
#                                transforms = get_transform(train=False))

In [ ]:
import json
success, json_annots = convert_to_coco_api(images_path=TEST_IMAGE_FOLDER, 
                                           annotations_path=TEST_MASK_FOLDER, 
                                           instance_segmentation = SEGMENTATION_MODEL)
with open('test_annotations.json', 'w') as file:
    json.dump(json_annots, file)

from references.detection.coco_utils import CocoDetection
# note that this is a modified version of torchvision.datasets.CocoDetection
test_dataset = CocoDetection(img_folder=TEST_IMAGE_FOLDER, ann_file='test_annotations.json', transforms=None)
# os.remove('test_annotations.json')

## Model Definition

In [ ]:
from transformers import RTDetrForObjectDetection

def get_rt_detr_model(
    id2label: Dict[int, str], 
):
    pre_trained_model_checkpoint: str = "PekingU/rtdetr_r50vd_coco_o365"
    label2id: Dict[str, int] =  {v: k for k, v in id2label.items()}
    model = RTDetrForObjectDetection.from_pretrained(
        pre_trained_model_checkpoint,
        id2label=id2label,
        label2id=label2id,
        anchor_image_size=None,
        ignore_mismatched_sizes=True,
    )
   
    # this is for freezing the backbone
    # for param in model.model.pixel_level_module.encoder.parameters():
    #     param.requires_grad_(False)

    return model

## Training
### Training parameters

In [ ]:
TRAIN_BATCH_SIZE = 16
OPTIMIZER = 'Adam' # can be set to 'SGD' as well for stochastic Gradient Descent
LEARNING_RATE = 1e-4
NUM_EPOCHS = 10
MODEL_PATH = 'checkpoints'

### Data loaders

It looks like Mask2Former can support overlapping instance masks (the dataset in the tutorial example had non-overlapping instance masks). I need to investigate further to be sure. But it seems, we do not need to define an order of objects when creating the instance masks for overlapping objects. So `collate_fn_2` below should be used. However, if the masks cannot be overlapping, We create the masks of overlapping objects in this order: bg, cage, cell, and then bead as cells can be inside cages (creating holes in cage masks), and beads can potentially be over the cells (creating holes). In this case, `collate_fn_1` should be used.

Note that `test_dataset` below is using a different class (`torchvision.datasets.CocoDetection` instead of `CellMaskDataset` defined above). We do not need any special collate function for the test_data_loader as we only use it for COCO evaluation.

In [ ]:
# we need to resize the input images (not needed as they are already in the correct 640 x 640 input size, and normalize
# them, we use the already implemented Hugging Face preprocessor for this conversion
# we pass all the other flags as False as the image is already augmented 
# index 0 will be used 
from transformers import RTDetrImageProcessor

hg_preprocessor = RTDetrImageProcessor(
    do_convert_annotations=True,
    do_resize=True,
    size={"width": MODEL_INPUT_SIZE, "height": MODEL_INPUT_SIZE},
    reduce_labels=False,
    do_rescale=True, 
    do_normalize=True
)

# it looks like Mas2Former can support overlapping instances, so there is no need to assign each pixel to only one class
# the function collate_fn_1 blow uses the function generate_instance_segmentation that only assigns each pixel to one class, 
# use collate_fn_2 that is more generic
def collate_fn(batch):
    processed_batch = []
    for img, formatted_annotations in batch:
        inputs = hg_preprocessor(images=img, annotations=formatted_annotations, return_tensors="pt")
        inputs = {k: v.squeeze() if isinstance(v, torch.Tensor) else v[0] for k,v in inputs.items()}
        processed_batch.append(inputs)

    data = {}
    data["pixel_values"] = torch.stack([x["pixel_values"] for x in processed_batch])
    data["labels"] = [x["labels"] for x in processed_batch]
    return data

# define training and validation data loaders
train_data_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size = TRAIN_BATCH_SIZE, shuffle = True, num_workers = 2,
    collate_fn = collate_fn)

test_data_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size = 1, shuffle = False, num_workers = 2,
    collate_fn = collate_fn)

# from references.detection.utils import collate_fn as collate_fn_test
# test_data_loader = torch.utils.data.DataLoader(
#     test_dataset, batch_size = 1, shuffle = False, num_workers = 2,
#     collate_fn = collate_fn_test)

print('Training data includes %d annotated images.' %len(train_dataset))
print('Test data includes %d annotated images.' %len(test_dataset))

### Visually checking some data

In [ ]:
import torchvision
COLORS = [(0, 0, 0), (0, 0, 255), (255, 0, 0), (0, 255, 0), (255, 255, 0), (255, 0, 255)]

def show_sample(idx, train = True, instance_segmentation = False):
    # pick the image from the data set
    if train:
        image, annotations = train_dataset[idx]
        boxes = np.array([record['bbox'] for record in annotations['annotations']]).astype(int)
        if len(boxes) > 0:
            boxes[:, 2] += boxes[:, 0]
            boxes[:, 3] += boxes[:, 1]
        labels  = np.array([record['category_id'] for record in annotations['annotations']]).astype(int)
        if instance_segmentation:
            masks = [record['mask'] for record in annotations['annotations']]
    else:
        image, annotations = test_dataset[idx]
        if isinstance(test_dataset, torchvision.datasets.CocoDetection):
            image = np.array(image)
            boxes = np.array([t['bbox'] for t in annotations['annotations']])
            if len(boxes) > 0:
                boxes[:, 2] += boxes[:, 0]
                boxes[:, 3] += boxes[:, 1]
            labels = np.array([record['category_id'] for record in annotations['annotations']])
            if instance_segmentation:
                masks = np.array([coco_mask_util.decode(record['segmentation']) for record in annotations['annotations']])
            
        else:
            boxes = np.array([record['bbox'] for record in annotations['annotations']]).astype(int)
            if len(boxes) > 0:
                boxes[:, 2] += boxes[:, 0]
                boxes[:, 3] += boxes[:, 1]
            labels  = np.array([record['category_id'] for record in annotations['annotations']]).astype(int)
            if instance_segmentation:
                masks = [record['mask'] for record in annotations['annotations']]
                
    for i in range(len(labels)):
        # the bounding box
        (xtl, ytl, xbr, ybr) = boxes[i]
        # use green color for masks
        color = COLORS[(labels[i] + 1) % len(COLORS)] # add 1 to be consistent with Mask R-CNN colors/labels
        if instance_segmentation:
            color_mask = color * np.repeat(np.expand_dims(masks[i][ytl:ybr, xtl:xbr], axis=2), 3, axis=2)
            blended = 0.4 * color_mask
            blended[color_mask == 0] = image[ytl:ybr, xtl:xbr][color_mask == 0]
            blended[color_mask > 0] += 0.6 * image[ytl:ybr, xtl:xbr][color_mask > 0]

            # store the blended ROI in the original image
            image[ytl:ybr, xtl:xbr] = blended.astype(np.uint8)
        
        if labels[i] in LABEL_MAP:
            text = LABEL_MAP[labels[i]]
        else:
            print('Incorrect ID was found %s' %labels[i])
            text = 'Unknown'
        
        # add label
        cv2.putText(image, text, (xtl, ytl + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        # add the bounding box with yellow color
        color = (255, 255, 0)
        cv2.rectangle(image, (xtl, ytl), (xbr, ybr), color, 1)
        
    print(f"Image size (W, H): {image.shape[1]}, {image.shape[0]}")
    # convert to PIL image to display
    return Image.fromarray(image)

In [ ]:
display(show_sample(184, False, SEGMENTATION_MODEL))

### Model selection
### From a pretrained model on COCO (scratch)

In [ ]:
model = get_rt_detr_model(
    id2label=LABEL_MAP
)

# train on the GPU or on the CPU, if a GPU is not available
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

print('Device available:' , device)

# move model to the right device
model.train()
model.to(device)

### From an already trained model on our dataset

In [ ]:
pre_trained_model_checkpoint: str = "checkpoints/checkpoint-543809"
model = RTDetrForObjectDetection.from_pretrained(
        pre_trained_model_checkpoint,
        id2label=LABEL_MAP,
        label2id={v: k for k, v in LABEL_MAP.items()},
        anchor_image_size=None,
        ignore_mismatched_sizes=False,
    )
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.eval()

### Evalutation function
A function to compute COCO mAP and mAR.

In [ ]:
from transformers.image_transforms import center_to_corners_format
from pycocotools.cocoeval import COCOeval

def to_cpu_device(tensor):
    """
    A function to move a CUDA torch input to CPU memory.
    Args:
        tensor (torch tensor).
    Returns:
        Moved to CPU.
    """
    return tensor.detach().cpu() if tensor.requires_grad else tensor.cpu()

def convert_bbox_yolo_to_pascal(boxes, image_size):
    """
    Convert bounding boxes from YOLO format (x_center, y_center, width, height) in range [0, 1]
    to Pascal VOC format (x_min, y_min, x_max, y_max) in absolute coordinates.

    Args:
        boxes (torch.Tensor): Bounding boxes in YOLO format
        image_size (Tuple[int, int]): Image size in format (height, width)

    Returns:
        torch.Tensor: Bounding boxes in Pascal VOC format (x_min, y_min, x_max, y_max)
    """
    # convert center to corners format
    boxes = center_to_corners_format(boxes)

    # convert to absolute coordinates
    height, width = image_size
    boxes = boxes * torch.tensor([[width, height, width, height]])

    return boxes

def convert_to_xywh(boxes):
    xmin, ymin, xmax, ymax = boxes.unbind(1)
    return torch.stack((xmin, ymin, xmax - xmin, ymax - ymin), dim=1)


def convert_preds_to_coco(predictions):
    coco_results = []
    for original_id, prediction in predictions.items():
        if len(prediction) == 0:
            continue
        
        boxes = prediction["boxes"]
        boxes = convert_to_xywh(boxes).tolist()
        
        scores = prediction["scores"].tolist()
        labels = prediction["labels"].tolist()
         
        coco_results.extend(
            [
                {
                    "image_id": original_id,
                    "category_id": labels[k],
                    "bbox": boxes[k],
                    "score": scores[k],
                }
                for k in range(len(scores))
            ]
        )
    return coco_results

from dataclasses import dataclass

@dataclass
class ModelOutput:
    logits: torch.Tensor
    pred_boxes: torch.Tensor


class MAPEvaluator:

    def __init__(self, data_loader, image_processor, threshold=0.4, max_dets=100):
        self.image_processor = image_processor
        self.threshold = threshold
        self.data_loader = data_loader
        self.all_predictions = []
        self.all_image_ids = []
        self.max_dets = max_dets

    def collect_image_sizes(self, batch):
        """Collect image sizes across a batch of the dataset as a list of batch_size size (list of 2 elements, height and width)."""
        batch_image_sizes = [to_cpu_device(x["size"]).numpy().tolist() for x in batch]
        return batch_image_sizes

    def collect_targets(self, batch_targets, batch_image_sizes):
        post_processed_batch_targets = []
        for target, size in zip(batch_targets, batch_image_sizes):
            boxes = to_cpu_device(target["boxes"])
            boxes = convert_bbox_yolo_to_pascal(boxes, size)
            labels = to_cpu_device(target["class_labels"])
            post_processed_batch_targets.append({"boxes": boxes, "labels": labels})
        return post_processed_batch_targets

    def collect_predictions(self, batch_predictions, batch_image_sizes):
        post_processed_predictions = []
        batch_logits, batch_boxes = batch_predictions[1], batch_predictions[2]
        output = ModelOutput(logits=batch_logits, pred_boxes=batch_boxes)
        # post_processed_output is a list of batch_size dictionaty elements, each dictionary containing the detections
        # for the images in the batch with keys as 'boxes', 'labels' and 'scores', and values as
        # - 'boxes': a (num_detection, 4) torch.float32 tensor of bounding boxes in (xtl, ytl, xbr, ybr) format
        # - 'labels': a (num_detection, 1) torch.int64 tensor of class IDs
        # - 'scores': a (num_detection, 1) torch.float32 tensor of detection confidences
        post_processed_output = self.image_processor.post_process_object_detection(
            output, threshold=self.threshold, target_sizes=batch_image_sizes
        )
        # move the detections to CPU
        post_processed_output = [{k: to_cpu_device(v) for k, v in outputs.items()} for outputs in post_processed_output]
        post_processed_predictions.extend(post_processed_output)
        return post_processed_predictions
    
    # metrics should be a dictionary with the following keys: 
    # 'map', 'map_50', 'map_75', 'map_small', 'map_medium', 'map_large', 'mar_1', 'mar_10', 'mar_100', 'mar_small', 'mar_medium', 'mar_large', 
    # 'map_cell', 'mar_100_cell', 'map_bead', 'mar_100_bead', 'map_soma', 'mar_100_soma'
    @torch.no_grad()
    def __call__(self, evaluation_results, compute_result):

        metrics = {
                'map': -1.0, 'map_50': -1.0, 'map_75': -1.0, 
                'map_small': -1.0, 'map_medium': -1.0, 'map_large': -1.0, 
                'mar_1': -1.0, 'mar_10': -1.0, 'mar_' + str(self.max_dets): -1.0, 
                'mar_small': -1.0, 'mar_medium': -1.0, 'mar_large': -1.0
            }
        
        batch_predictions, batch_targets = evaluation_results.predictions, evaluation_results.label_ids
        
        batch_image_sizes = self.collect_image_sizes(batch_targets)
        post_processed_batch_targets = self.collect_targets(batch_targets, batch_image_sizes)
        post_processed_batch_predictions = self.collect_predictions(batch_predictions, batch_image_sizes)
        results = {int(target["image_id"].item()): output for target, output in zip(batch_targets, post_processed_batch_predictions)}
        results = convert_preds_to_coco(results)
        self.all_predictions.extend(results)
        self.all_image_ids += [int(target["image_id"].item()) for target in batch_targets]
        
        if compute_result:
            n_threads = torch.get_num_threads()
            # FIXME remove this and make paste_masks_in_image run on the GPU
            torch.set_num_threads(1)
            cpu_device = torch.device("cpu")
    
            coco_gt = self.data_loader.dataset.coco   
            coco_dt = coco_gt.loadRes(self.all_predictions)  # init predictions api
    
            evaluator_time = time.time()
    
            # bounding box evaluation
            coco_evaluator_bbox = COCOeval(coco_gt, coco_dt, "bbox")
            coco_evaluator_bbox.params.maxDets = [1, 10, self.max_dets]
            coco_evaluator_bbox.params.imgIds = self.all_image_ids
            coco_evaluator_bbox.evaluate()
            coco_evaluator_bbox.accumulate()
            coco_evaluator_bbox.summarize()
            evaluator_time = time.time() - evaluator_time
    
            print("evaluator_time:", evaluator_time)

            torch.set_num_threads(n_threads)
            for i, key in enumerate(metrics.keys()):
                metrics[key] = coco_evaluator_bbox.stats[i]
            
            metrics = {k: round(v.item(), 4) for k, v in metrics.items()}

            # clear up the history
            self.all_predictions = []
            self.all_image_ids = []
        
        return metrics

eval_compute_metrics_fn = MAPEvaluator(data_loader=test_data_loader, image_processor=hg_preprocessor, threshold=0.4, max_dets=100)

### Run Training using Hugging face training script

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=MODEL_PATH,
    num_train_epochs=NUM_EPOCHS,
    max_grad_norm=0.1,
    learning_rate=LEARNING_RATE,
    warmup_steps=300,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    torch_empty_cache_steps=int(len(train_dataset) / (5 * TRAIN_BATCH_SIZE)), 
    batch_eval_metrics=True,
    dataloader_num_workers=2,
    metric_for_best_model="eval_map",
    greater_is_better=True,
    load_best_model_at_end=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    remove_unused_columns=False,
    eval_do_concat_batches=False,
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=hg_preprocessor,
    data_collator=collate_fn,
    compute_metrics=eval_compute_metrics_fn,
)

trainer.train()

In [ ]:
# Epoch	Training Loss	Validation Loss	Map	Map 50	Map 75	Map Small	Map Medium	Map Large	Mar 1	Mar 10	Mar 100	Mar Small	Mar Medium	Mar Large
# 1	8.449600	9.477483	0.453100	0.705400	0.533500	0.283200	0.465200	0.212800	0.157800	0.492900	0.565200	0.344000	0.602500	0.315500
# 2	8.223800	8.733813	0.507400	0.760000	0.607400	0.280900	0.502300	0.302800	0.177600	0.528700	0.603200	0.350800	0.608500	0.439900
# 3	8.031400	11.996222	0.396600	0.567000	0.475000	0.275600	0.403500	0.157200	0.101200	0.375000	0.448000	0.308800	0.463400	0.178200
# 4	7.950800	7.612844	0.504700	0.726500	0.611500	0.294000	0.520800	0.313600	0.158200	0.498000	0.568800	0.339400	0.591400	0.378500
# 5	7.697000	11.295392	0.341100	0.472200	0.414900	0.292200	0.351300	0.094000	0.082000	0.321900	0.400900	0.336800	0.410300	0.103900
# 6	7.596000	10.256644	0.424500	0.596800	0.505400	0.282700	0.425000	0.217800	0.115500	0.406500	0.480600	0.330500	0.480500	0.243900
# 7	7.374200	9.024280	0.534400	0.745000	0.632700	0.315200	0.520700	0.328600	0.167600	0.534500	0.612600	0.361800	0.611800	0.406600
# 8	7.285000	10.126621	0.481100	0.684100	0.571400	0.292400	0.471100	0.278200	0.144400	0.470500	0.542500	0.345000	0.532300	0.315900
# 9	7.090000	10.538335	0.491200	0.689400	0.585500	0.281600	0.485300	0.337100	0.151500	0.481900	0.552100	0.318300	0.555200	0.383000


# 0.360800	0.498800	0.428600	0.128100	0.383700	0.324000	0.156000	0.396000	0.413800	0.143100	0.441000	0.372300


# load the model, the latest saved checkpoint will be loaded
pre_trained_model_checkpoint: str = "checkpoints/checkpoint-543809"
model = RTDetrForObjectDetection.from_pretrained(
        pre_trained_model_checkpoint,
        id2label=LABEL_MAP,
        label2id={v: k for k, v in LABEL_MAP.items()},
        anchor_image_size=None,
        ignore_mismatched_sizes=False,
    )

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)


DETECTION_REMAP = None
# DETECTION_REMAP = {"cell-adhered": "cell", "soma": "cell"}


RESIZE: Final[Dict[Tuple[int, int, str], Tuple[int, int]]] = {
    (2000, 1600, "10x"): (1000, 800),
    (4512, 4512, "10x"): (2440, 2440),
    (4512, 4512, "4x"): (4512, 4512),
}
# A dictionary with keys as the input (original) image size (width, height, magnification)
# tuple and values as the list of coordinates (xtl, ytl, xbr, ybr) of sub-images/crops
# to run YOLOv5 on each
# note that the crop coordinates are with respect to resized image dimensions specified above
CROP_CORNERS: Final[Dict[Tuple[int, int, str], List[List[int]]]] = {
    (2000, 1600, "10x"): [
        [0, 0, 640, 640],
        [0, 160, 640, 800],
        [360, 0, 1000, 640],
        [360, 160, 1000, 800],
    ],
    (4512, 4512, "10x"): [
        [0, 0, 640, 640],
        [0, 600, 640, 1240],
        [0, 1200, 640, 1840],
        [0, 1800, 640, 2440],
        [600, 0, 1240, 640],
        [600, 600, 1240, 1240],
        [600, 1200, 1240, 1840],
        [600, 1800, 1240, 2440],
        [1200, 0, 1840, 640],
        [1200, 600, 1840, 1240],
        [1200, 1200, 1840, 1840],
        [1200, 1800, 1840, 2440],
        [1800, 0, 2440, 640],
        [1800, 600, 2440, 1240],
        [1800, 1200, 2440, 1840],
        [1800, 1800, 2440, 2440],
    ],
    (4512, 4512, "4x"): [
        [0, 0, 640, 640],
        [0, 560, 640, 1200],
        [0, 1120, 640, 1760],
        [0, 1680, 640, 2320],
        [0, 2240, 640, 2880],
        [0, 2800, 640, 3440],
        [0, 3360, 640, 4000],
        [0, 3872, 640, 4512],
        [560, 0, 1200, 640],
        [560, 560, 1200, 1200],
        [560, 1120, 1200, 1760],
        [560, 1680, 1200, 2320],
        [560, 2240, 1200, 2880],
        [560, 2800, 1200, 3440],
        [560, 3360, 1200, 4000],
        [560, 3872, 1200, 4512],
        [1120, 0, 1760, 640],
        [1120, 560, 1760, 1200],
        [1120, 1120, 1760, 1760],
        [1120, 1680, 1760, 2320],
        [1120, 2240, 1760, 2880],
        [1120, 2800, 1760, 3440],
        [1120, 3360, 1760, 4000],
        [1120, 3872, 1760, 4512],
        [1680, 0, 2320, 640],
        [1680, 560, 2320, 1200],
        [1680, 1120, 2320, 1760],
        [1680, 1680, 2320, 2320],
        [1680, 2240, 2320, 2880],
        [1680, 2800, 2320, 3440],
        [1680, 3360, 2320, 4000],
        [1680, 3872, 2320, 4512],
        [2240, 0, 2880, 640],
        [2240, 560, 2880, 1200],
        [2240, 1120, 2880, 1760],
        [2240, 1680, 2880, 2320],
        [2240, 2240, 2880, 2880],
        [2240, 2800, 2880, 3440],
        [2240, 3360, 2880, 4000],
        [2240, 3872, 2880, 4512],
        [2800, 0, 3440, 640],
        [2800, 560, 3440, 1200],
        [2800, 1120, 3440, 1760],
        [2800, 1680, 3440, 2320],
        [2800, 2240, 3440, 2880],
        [2800, 2800, 3440, 3440],
        [2800, 3360, 3440, 4000],
        [2800, 3872, 3440, 4512],
        [3360, 0, 4000, 640],
        [3360, 560, 4000, 1200],
        [3360, 1120, 4000, 1760],
        [3360, 1680, 4000, 2320],
        [3360, 2240, 4000, 2880],
        [3360, 2800, 4000, 3440],
        [3360, 3360, 4000, 4000],
        [3360, 3872, 4000, 4512],
        [3872, 0, 4512, 640],
        [3872, 560, 4512, 1200],
        [3872, 1120, 4512, 1760],
        [3872, 1680, 4512, 2320],
        [3872, 2240, 4512, 2880],
        [3872, 2800, 4512, 3440],
        [3872, 3360, 4512, 4000],
        [3872, 3872, 4512, 4512],
    ],
}

model_param_dict = {}
model_param_dict['model_state_dict'] = model.state_dict()
model_param_dict['label_map'] = LABEL_MAP
model_param_dict['resize_dict'] = RESIZE
model_param_dict['crop_corners_dict'] = CROP_CORNERS
if DETECTION_REMAP is not None:
    model_param_dict['detected_class_names_remap'] = DETECTION_REMAP

torch.save(model_param_dict, os.path.join(MODEL_PATH, 'final.pt'))

In [ ]:
from pprint import pprint

metrics = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="eval")
pprint(metrics)

### Run Inference
To run inference on a previously trained model, run the cells above up to "Cell size analysis".

In [ ]:
from torchvision.transforms import functional as F
from transformers import RTDetrImageProcessor

def to_numpy(tensor):
    """
    A function to convert a torch input to numpy array.
    Args:
        tensor (torch tensor).
    Returns:
        Converted to numpy array.
    """
    return tensor.detach().cpu().numpy() if tensor.requires_grad else tensor.cpu().numpy()

def show_predictions(image_pil, predictions, color_depth=12):
    # convert to a numpy array
    image = np.array(image_pil)
    # scale
    image = (255 * image.astype(float) / (2**color_depth - 1)).astype(np.uint8)
    # convert to 3-channels
    image = np.repeat(np.expand_dims(image, axis=2), 3, axis=2)

    boxes = predictions['boxes']
    labels = predictions['labels']

    for i in range(len(boxes)):
        # the bounding box
        (xtl, ytl, xbr, ybr) = boxes[i].astype(int)
        # use green color for masks
        color = COLORS[(labels[i] + 1) % len(COLORS)]
        
        if labels[i] in LABEL_MAP:
            text = LABEL_MAP[labels[i]]
        else:
            print('Incorrect ID was found %s' %labels[i])
            text = 'Unknown'
        
        # add label
        cv2.putText(image, text, (xtl, ytl + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        cv2.rectangle(image, (xtl, ytl), (xbr, ybr), color, 1)
        
    # convert to PIL image to display
    return Image.fromarray(image)

# we return the results in the same format as Mask R-CNN and YOLO to be able to reuse the codes written for that model
def predict_batch(model, input_images_list, device):
    
    model.eval()
    model.to(device)

    detection_threshold: float = 0.4

    # convert to 3-channel images if needed, and store the original image dimensions for 
    # post processing
    images_list: List[np.array] = []
    org_img_dims: List[Tuple[int, int]] = []
    
    for img in input_images_list:
        img_shape: tuple = img.shape
        if len(img_shape) < 3:
            images_list.append(cv2.cvtColor(img, cv2.COLOR_GRAY2RGB))
        else:
            images_list.append(img)
        org_img_dims.append(img_shape[:2])
    
    hg_preprocessor = RTDetrImageProcessor(
        do_convert_annotations=True,
        do_resize=True,
        size={"width": MODEL_INPUT_SIZE, "height": MODEL_INPUT_SIZE},
        reduce_labels=False,
        do_rescale=True, 
        do_normalize=True
    )
       
    processed_imgs_dict = hg_preprocessor(images_list, return_tensors="pt")
    with torch.no_grad():
        outputs = model(pixel_values=processed_imgs_dict["pixel_values"].to(device))
        
        processed_outputs = hg_preprocessor.post_process_object_detection(
            outputs, threshold=detection_threshold, target_sizes=org_img_dims
        )

    # processed_outputs is a list of len(input_images_list) dictionary elements, each dictionary containing the detections
    # for the input image in the input list with keys as 'boxes', 'labels' and 'scores', and values as
    # - 'boxes': a (num_detection, 4) torch.float32 tensor of bounding boxes in (xtl, ytl, xbr, ybr) format
    # - 'labels': a (num_detection, 1) torch.int64 tensor of class IDs
    # - 'scores': a (num_detection, 1) torch.float32 tensor of detection confidences
        
    if len(processed_outputs) == 0:
        # this should not happen and is not expected, return as if the model has not detected anything (for the whole list of images)
        return [
            {'boxes': [],
             'labels': [],
             'scores': [],
            }
        ] * len(images_list)

    # move to CPU and convert to numpy arrays before returning
    return [{k: to_numpy(v) for k, v in result.items()} for result in processed_outputs]

In [ ]:
pre_trained_model_checkpoint: str = "checkpoints/checkpoint-543809"
model = RTDetrForObjectDetection.from_pretrained(
        pre_trained_model_checkpoint,
        id2label=LABEL_MAP,
        label2id={v: k for k, v in LABEL_MAP.items()},
        anchor_image_size=None,
        ignore_mismatched_sizes=False,
    )
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.eval()

In [ ]:
# idx = 145123
idx = 3241
# img_path = os.path.join(test_dataset.images_path, test_dataset.imgs[idx])
img_id = test_dataset.coco.getImgIds()[idx]
img_path = os.path.join(test_dataset.root, test_dataset.coco.imgs[img_id]['file_name'])
img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)

In [ ]:
out = predict_batch(model, [img], device)[0]

In [ ]:
display(show_predictions(img, out, 8))

In [ ]:
display(show_sample(idx, False))

### Measuring the run-time

In [ ]:
import time
start = time.time()
for i in range(10):
    out = predict_batch(model, [img], device)[0]
print('Running RT-DETR took {} ms'.format((time.time() - start) * 100))